# Глава 7. Признаки, метрики и Pipeline — практика в Jupyter

Те же задачи, что на сайте курса, но в своём Jupyter: так вы привыкаете к блокноту, горячим клавишам и своему окружению. Проверка — та же, что в браузере.

**Как запустить** (один раз поставьте [uv](https://docs.astral.sh/uv/), см. урок «Своё рабочее место — uv и Jupyter» главы 2). В терминале, в папке курса:

```sh
uv run --with-requirements site/notebooks/requirements.txt jupyter lab site/notebooks
```

Без uv: `python3 -m venv .venv`, затем `.venv/bin/pip install -r site/notebooks/requirements.txt` (в Windows — `.venv\Scripts\pip`) и `.venv/bin/jupyter lab site/notebooks`. Откройте этот файл в папке `site/notebooks` — рядом с ним лежит расширение `dsc.py`, без него проверка не заработает.

**Как решать**

1. Запустите ячейку ниже с `%load_ext dsc` — она подключает проверку. После перезапуска ядра запустите её снова.
2. В ячейке задачи первая строка — `%%task глава/урок/id`. Её не трогайте, пишите код ниже и запускайте **Shift+Enter**. Под выводом появится «✓ Верно» или подсказка, что не так.
3. Переменные живут между ячейками, как в любом блокноте. Исключение — переменные, в которые задача просит положить ответ: перед запуском ячейки они стираются, чтобы проверка видела результат именно этого запуска. Подготовка задачи (если она есть) выполняется перед вашим кодом при каждом запуске.
4. Застряли — `dsc.hint('id')` покажет следующую подсказку, `dsc.solution('id', sure=True)` — эталон с разбором. `dsc.progress()` — что решено в этом сеансе.
5. Файлы данных берутся по имени, как на сайте: `pd.read_csv('orders.csv')`. Код работает в папке `site/notebooks/_work` — там копия `site/data`, и туда же попадает всё, что сохраняет ваш код.

Прогресс сайта и блокнота не связаны: отметки о решённых задачах на сайте ставятся только там. Чтобы начать с чистого листа, скачайте блокнот заново или возьмите его из архива курса.

In [ ]:
%load_ext dsc

## Урок 1. Матрица ошибок

Лес для оттока — такой же, как в проекте главы 6, — прав почти в 89 % случаев. Звучит отлично — пока не выясняется, что модель «никто не уйдёт» права в 78 %. Учимся раскладывать ответы модели на четыре исхода, считать их одной функцией и понимать, какие ошибки модель делает и сколько они стоят «Зерну».

Теория — на сайте курса: `ch07/01-confusion.html`. Задачи «собери код из строк» (1) остались на сайте.

### Четыре исхода руками · разминка

Отдел лояльности отправил бонусы по прогнозу модели десяти гостям и через 60 дней узнал правду. Посчитайте без sklearn, масками pandas, четыре исхода: `tp`, `fp`, `fn`, `tn` (целые числа). В `acc` сохраните долю правильных ответов (2 знака), посчитанную по этим четырём числам.

*Подсказки (2): `dsc.hint('count-four')`*

In [ ]:
%%task ch07/01-confusion/count-four
import pandas as pd

fact = pd.Series([1, 0, 0, 1, 1, 0, 0, 0, 1, 0])
pred = pd.Series([1, 0, 1, 0, 1, 0, 0, 1, 0, 0])

### Сколько стоит 78 % · разминка

Обучите `DummyClassifier(strategy="most_frequent")` и сохраните:

- `acc_dummy` — её accuracy на тесте (3 знака);
- `churn_test` — долю ушедших в тесте (3 знака);
- `found` — сколько ушедших гостей теста базовая модель нашла (целое число, по матрице ошибок).

*Подсказки (2): `dsc.hint('dummy-acc')`*

In [ ]:
%%task ch07/01-confusion/dummy-acc
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Матрица леса · база

Обучите `RandomForestClassifier(n_estimators=50, random_state=0)` и сохраните его матрицу ошибок на тесте в `cm`, а её клетки — в `tn`, `fp`, `fn`, `tp` (целые числа).

*Подсказки (2): `dsc.hint('forest-cm')`*

In [ ]:
%%task ch07/01-confusion/forest-cm
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Правило 45 дней · база

Отдел лояльности считает гостя уходящим, если `days_since_last > 45`. Превратите правило в прогноз для тестовых гостей (1 — уйдёт, 0 — останется) и сохраните:

- `rule_cm` — матрицу ошибок правила;
- `rule_missed` — сколько ушедших гостей правило пропустило (целое);
- `rule_extra` — сколько лишних бонусов оно бы раздало (целое).

*Подсказки (2): `dsc.hint('rule-cm')`*

In [ ]:
%%task ch07/01-confusion/rule-cm
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Доли ошибок · база

Для леса `RandomForestClassifier(n_estimators=50, random_state=0)` посчитайте на тесте (все — 3 знака):

- `missed_share` — долю пропущенных среди ушедших гостей;
- `extra_share` — долю лишних бонусов среди оставшихся гостей;
- `error_rate` — долю всех ошибок среди всех гостей теста.

*Подсказки (2): `dsc.hint('ch07/01-confusion/shares')`*

In [ ]:
%%task ch07/01-confusion/shares
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Осторожное и смелое дерево · база

Сравните два дерева: `DecisionTreeClassifier(max_depth=2, random_state=0)` и `DecisionTreeClassifier(max_depth=3, random_state=0)`. Сохраните:

- `acc2`, `acc3` — accuracy на тесте (3 знака);
- `tp2`, `tp3` — сколько уходящих нашло каждое дерево (целые);
- `fp2`, `fp3` — сколько лишних бонусов раздало каждое (целые);
- `more_found` — строку `"2"` или `"3"`: какое дерево нашло больше уходящих.

*Подсказки (2): `dsc.hint('two-trees')`*

In [ ]:
%%task ch07/01-confusion/two-trees
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Сколько стоят ошибки · посложнее

Отдел лояльности оценил: бонус стоит 300 ₽ (платим за каждого гостя, которому его отправили, — и за TP, и за FP), а каждый пропущенный уходящий гость — это 2000 ₽ потерянной выручки (в «Разберём вместе» мы брали 3 000 ₽; отдел пересчитал и решил, что для наших гостей это 2000 ₽ — схема та же, другая цена). Посчитайте стоимость `300 × (TP + FP) + 2000 × FN` на тесте для четырёх вариантов и сохраните словарь `costs` с ключами:

- `"никто"` — `DummyClassifier(strategy="most_frequent")`;
- `"правило"` — `days_since_last > 45`;
- `"дерево"` — `DecisionTreeClassifier(max_depth=2, random_state=0)`;
- `"лес"` — `RandomForestClassifier(n_estimators=50, random_state=0)`.

Значения — целые рубли. В `cheapest` сохраните ключ самого дешёвого варианта.

*Подсказки (2): `dsc.hint('cost')`*

In [ ]:
%%task ch07/01-confusion/cost
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Кого модель пропускает · со звёздочкой

Разбор ошибок — лучший способ понять модель. Возьмите лес `RandomForestClassifier(n_estimators=50, random_state=0)` и **только ушедших** гостей теста. Разделите их на найденных (TP) и пропущенных (FN) и сравните группы. Сохраните:

- `days_tp`, `days_fn` — медиану `days_since_last` у найденных и у пропущенных (1 знак);
- `visits_tp`, `visits_fn` — медиану `visits_90d` (1 знак);
- `new_tp`, `new_fn` — долю новичков (`tenure_months <= 3`) в каждой группе (3 знака).

*Подсказки (2): `dsc.hint('missed-who')` · задача по желанию*

In [ ]:
%%task ch07/01-confusion/missed-who
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 2. Precision, recall и F1

Матрица ошибок — четыре числа, а сравнивать модели удобнее по одному-двум. Учимся задавать матрице два вопроса бизнеса — «бонусы ушли по адресу?» и «скольких уходящих нашли?», — сводить ответы в F1 и решать, какая ошибка дороже.

Теория — на сайте курса: `ch07/02-precision-recall.html`. Задачи «собери код из строк» (1) остались на сайте.

### Метрики по формулам · разминка

У леса из прошлого урока на тесте TP = 90, FP = 23, FN = 44. Посчитайте по формулам, без sklearn (все — 3 знака):

- `prec` — точность;
- `rec` — полноту;
- `f1` — F1 по точности и полноте (из неокруглённых значений).

*Подсказки (2): `dsc.hint('ch07/02-precision-recall/by-hand')`*

In [ ]:
%%task ch07/02-precision-recall/by-hand
tp, fp, fn = 90, 23, 44

### Точность 100 % · разминка

Модель отметила для акции одного гостя из двенадцати — и угадала. Посчитайте функциями sklearn `prec`, `rec` и `f1` (3 знака).

*Подсказки (2): `dsc.hint('perfect-prec')`*

In [ ]:
%%task ch07/02-precision-recall/perfect-prec
fact = [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0]
pred = [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]

### Метрики леса · база

Обучите `RandomForestClassifier(n_estimators=50, random_state=0)` и сохраните для теста:

- `prec`, `rec`, `f1` — точность, полноту и F1 (3 знака);
- `bonuses` — сколько гостей теста получат бонус по прогнозу модели (целое).

*Подсказки (2): `dsc.hint('forest-prf')`*

In [ ]:
%%task ch07/02-precision-recall/forest-prf
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Таблица трёх подходов · база

Сравните на тесте правило `days_since_last > 45`, `DecisionTreeClassifier(max_depth=2, random_state=0)` и `RandomForestClassifier(n_estimators=50, random_state=0)`. Сохраните таблицу `scores`: индекс — `"правило"`, `"дерево"`, `"лес"`, столбцы `precision`, `recall`, `f1` (3 знака) и `cost` — потери в рублях по схеме урока «Матрица ошибок»: бонус 300 ₽ платим всем, кому отправили (TP и FP), пропущенный уходящий (FN) стоит 2 000 ₽. В `best_f1` — название строки с наибольшей F1, в `best_cost` — с наименьшими потерями.

*Подсказки (2): `dsc.hint('three-models')`*

In [ ]:
%%task ch07/02-precision-recall/three-models
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Бонус всем · база

Проверьте на тесте «модель», которая отправляет бонус **всем** гостям: прогноз — единица для каждого гостя `y_test`. Сохраните (3 знака):

- `prec_all`, `rec_all` — её точность и полноту;
- `mean_all` — обычное среднее точности и полноты;
- `f1_all` — её F1.

*Подсказки (2): `dsc.hint('all-bonus')`*

In [ ]:
%%task ch07/02-precision-recall/all-bonus
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Отчёт по классам · база

Для `RandomForestClassifier(n_estimators=50, random_state=0)` получите `classification_report` словарём (`output_dict=True`) и сохраните из него:

- `rec1` — полноту для класса 1 (3 знака);
- `rec0` — полноту для класса 0 (3 знака);
- `support1` — сколько ушедших в тесте (целое);
- `macro_f1` — F1 из строки `macro avg` (3 знака).

*Подсказки (2): `dsc.hint('ch07/02-precision-recall/report')`*

In [ ]:
%%task ch07/02-precision-recall/report
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### План по полноте · посложнее

Отдел лояльности ставит условие: модель должна находить **не меньше 70 %** уходящих гостей, а среди таких моделей нужна самая точная. Переберите деревья `DecisionTreeClassifier(max_depth=d, random_state=0)` для `d` от 1 до 8 и сохраните:

- `trees` — DataFrame со столбцами `depth`, `precision`, `recall`, `f1` (по строке на дерево, метрики на тесте, 3 знака);
- `best_depth` — глубину дерева, которое выполняет условие по полноте и среди таких имеет наибольшую точность (целое).

*Подсказки (2): `dsc.hint('recall-plan')`*

In [ ]:
%%task ch07/02-precision-recall/recall-plan
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Кого лес ловит хуже · посложнее

В задаче со звёздочкой урока «Матрица ошибок» (missed-who) мы сравнили найденных и пропущенных уходящих: среди пропущенных новичков (`tenure_months <= 3`) вдвое больше — 26 % против 13 %. Проверьте это метриками. Для `RandomForestClassifier(n_estimators=50, random_state=0)` посчитайте точность и полноту отдельно для новичков теста (`tenure_months <= 3`) и для остальных гостей теста (3 знака): `prec_new`, `rec_new`, `prec_old`, `rec_old`.

*Подсказки (2): `dsc.hint('newcomers')`*

In [ ]:
%%task ch07/02-precision-recall/newcomers
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Три меры — три победителя · со звёздочкой

В теории мы упомянули F-beta: `fbeta_score(y_true, y_pred, beta=...)` из `sklearn.metrics`. При `beta=2` полнота весит больше точности, при `beta=0.5` — наоборот, при `beta=1` это F1.

Для деревьев `DecisionTreeClassifier(max_depth=d, random_state=0)`, `d` от 1 до 8, посчитайте на тесте F1, F2 (`beta=2`) и F0.5 (`beta=0.5`). Сохраните целые числа:

- `d_f1` — глубину с наибольшей F1;
- `d_f2` — глубину с наибольшей F2;
- `d_f05` — глубину с наибольшей F0.5.

*Подсказки (2): `dsc.hint('f-beta')` · задача по желанию*

In [ ]:
%%task ch07/02-precision-recall/f-beta
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 3. Порог, PR- и ROC-кривые

Модель выдаёт вероятность ухода, а решение «дать бонус или нет» принимает порог. Учимся двигать порог и видеть, как меняются точность и полнота, рисовать PR- и ROC-кривые и понимать, что на самом деле измеряет ROC-AUC. Как выбрать порог по деньгам — в следующем уроке.

Теория — на сайте курса: `ch07/03-threshold.html`. Задачи «собери код из строк» (1) остались на сайте.

### Порог 0.3 · разминка

Отдел лояльности решил давать бонус всем, у кого вероятность ухода 0.3 и выше. Посчитайте для тестовых гостей: `n_bonus` — сколько бонусов раздадут (целое число), `prec` и `rec` — точность и полноту при таком пороге (3 знака).

*Подсказки (2): `dsc.hint('thr-30')`*

In [ ]:
%%task ch07/03-threshold/thr-30
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
p = model.predict_proba(X_test)[:, 1]

### TPR и FPR · разминка

Для порога 0.4 разложите матрицу ошибок на тесте в четыре числа `tn, fp, fn, tp` (урок «Матрица ошибок») и посчитайте `tpr` — долю пойманных уходящих и `fpr` — долю оставшихся гостей, которым зря дали бонус (обе — 3 знака). Это одна точка ROC-кривой.

*Подсказки (2): `dsc.hint('rates')`*

In [ ]:
%%task ch07/03-threshold/rates
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Порог под план по полноте · база

Отдел лояльности хочет поймать **не меньше 85 %** уходящих гостей. По PR-кривой леса на тесте найдите порог, который при этом даёт самую высокую точность. Сохраните `best_t` — этот порог (3 знака), `best_prec` — точность при нём (3 знака) и `curve` — таблицу DataFrame со столбцами `threshold`, `precision`, `recall` (все пороги кривой).

*Подсказки (2): `dsc.hint('recall80')`*

In [ ]:
%%task ch07/03-threshold/recall80
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Нарисовать ROC-кривую · база

Нарисуйте ROC-кривую леса на тесте: первая линия на графике — сама кривая (`fpr` по X, `tpr` по Y), вторая — диагональ от (0, 0) до (1, 1). Подпишите оси (любой текст) и дайте заголовок, в котором есть слово `ROC`. Сохраните `auc` — ROC-AUC леса на тесте (3 знака).

*Подсказки (2): `dsc.hint('roc-plot')`*

In [ ]:
%%task ch07/03-threshold/roc-plot
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_curve, roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Три модели по AUC · база

Сравните три модели по ROC-AUC на тесте: базовую `DummyClassifier(strategy="most_frequent")` из урока «Матрица ошибок», `LogisticRegression(max_iter=2000)` и `RandomForestClassifier(n_estimators=100, random_state=0)`. Сохраните `auc_dummy`, `auc_lr`, `auc_rf` (3 знака) и в `best` — строку `"dummy"`, `"lr"` или `"rf"` — у какой модели AUC выше.

*Подсказки (2): `dsc.hint('three-auc')`*

In [ ]:
%%task ch07/03-threshold/three-auc
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### AUC через пары · посложнее

Посчитайте ROC-AUC леса на тесте **без** `roc_auc_score` — по определению через пары. Переберите **все** пары «ушедший гость — оставшийся гость»: пара засчитывается за 1, если оценка ушедшего выше, и за 0.5, если оценки равны. Сохраните `n_pairs` — число пар, `auc_pairs` — долю «выигранных» пар (3 знака). Для сверки сохраните `auc_sk` — то же через `roc_auc_score` (3 знака).

*Подсказки (2): `dsc.hint('ch07/03-threshold/pairs')`*

In [ ]:
%%task ch07/03-threshold/pairs
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]
left = p[y_test == 1]
stayed = p[y_test == 0]

## Урок 4. Порог по деньгам

Точность и полнота — промежуточные цифры, бизнесу важнее деньги. Считаем прибыль акции удержания для каждого порога, находим лучший, укладываемся в бюджет на бонусы — и выбираем порог честно, не подглядывая в тест.

Теория — на сайте курса: `ch07/03b-money-threshold.html`.

### Прибыль при одном пороге · разминка

При пороге 0.5 лес из теории раздаёт 116 бонусов, из них 91 достаётся уходящим (TP). Всего уходящих в тесте 134. Цены — как в теории: бонус 200 ₽, выгода от бонуса уходящему 1000 ₽. Посчитайте руками (кодом, но без модели):

- `profit` — прибыль акции: TP × 1000 − (число бонусов) × 200;
- `loss` — потери по схеме урока «Матрица ошибок»: (число бонусов) × 200 + FN × 1000, где FN — пропущенные уходящие.

*Подсказки (2): `dsc.hint('warm-profit')`*

In [ ]:
%%task ch07/03b-money-threshold/warm-profit
tp = 91
bonuses = 116
churned = 134
COST, GAIN = 200, 1000

### Прибыль при дорогом бонусе · база

Бонус подорожал: теперь он стоит **300 ₽**, а выгода от бонуса уходящему гостю — **900 ₽**. Для порогов `[0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]` посчитайте прибыль на тесте: TP × 900 − (число бонусов) × 300, бонус — при p ≥ порога. Сохраните `profits` — **список** прибылей (целые числа) в порядке порогов, `best_t` — порог с наибольшей прибылью и `best_profit` — эту прибыль.

*Подсказки (2): `dsc.hint('ch07/03b-money-threshold/profit')`*

In [ ]:
%%task ch07/03b-money-threshold/profit
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8]

### Бюджет на 100 бонусов · посложнее

Бюджета хватает примерно на 100 бонусов на тестовую группу. Возьмите в качестве порога `t100` вероятность 100-го по риску гостя (отсортируйте `p` по убыванию). Бонус получают все с `p >= t100` — из-за одинаковых вероятностей их может оказаться чуть больше 100. Сохраните `t100` (3 знака), `n_bonus` — сколько бонусов раздадут, `prec` и `rec` — точность и полноту при этом пороге (3 знака).

*Подсказки (2): `dsc.hint('budget')`*

In [ ]:
%%task ch07/03b-money-threshold/budget
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
p = RandomForestClassifier(n_estimators=100, random_state=0).fit(X_train, y_train).predict_proba(X_test)[:, 1]

### Честный выбор порога · со звёздочкой

В теории мы выбрали порог по прибыли прямо на тесте — это подглядывание. Сделайте честно:

1. Разделите `X_train, y_train` ещё раз: `train_test_split(X_train, y_train, test_size=0.25, random_state=0, stratify=y_train)` → `X_fit, X_val, y_fit, y_val`.
2. Обучите `RandomForestClassifier(n_estimators=100, random_state=0)` на `X_fit`.
3. На `X_val` посчитайте прибыль (бонус 200 ₽, выгода 1000 ₽) для порогов `np.arange(0.05, 1, 0.05).round(2)` и выберите лучший — `t_val` (если лучших несколько, берите первый).
4. Той же моделью оцените тест: `profit_test` — прибыль на тесте при пороге `t_val`, `profit_05` — при пороге 0.5 (целые числа).

*Подсказки (2): `dsc.hint('honest-t')` · задача по желанию*

In [ ]:
%%task ch07/03b-money-threshold/honest-t
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 5. Дисбаланс классов

Уходящих гостей в данных «Зерна» лишь около 22 %, и модель по умолчанию тянется к большинству: недооценивает отток и пропускает уходящих. Учимся выравнивать классы весами, размножением и прореживанием строк, не устраивать при этом утечку и понимать, что балансировка на самом деле меняет, а что нет.

Теория — на сайте курса: `ch07/04-imbalance.html`. Задачи «собери код из строк» (1) остались на сайте.

### Веса руками · разминка

Посчитайте веса классов, которые выставит `class_weight="balanced"` для обучающей части: `w0` — вес оставшихся гостей, `w1` — вес уходящих (оба — 3 знака). Формула: n / (2 × n_класса).

*Подсказки (2): `dsc.hint('w-hand')`*

In [ ]:
%%task ch07/04-imbalance/w-hand
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Модель недооценивает отток · разминка

Обучите `LogisticRegression(max_iter=2000)` без весов и сохраните: `share_true` — долю ушедших в тесте, `share_pred` — долю тестовых гостей, которым модель (`predict`) отвечает «уйдёт» (обе — 3 знака).

*Подсказки (2): `dsc.hint('share-pred')`*

In [ ]:
%%task ch07/04-imbalance/share-pred
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Взвешенная логистическая регрессия · база

Обучите `LogisticRegression(max_iter=2000, class_weight="balanced")` и посчитайте на тесте `prec`, `rec`, `f1` (по ответам `predict`) и `auc` (по вероятностям) — все с 3 знаками. Затем сохраните `auc_plain` — ROC-AUC такой же модели **без** весов (3 знака).

*Подсказки (2): `dsc.hint('lr-bal')`*

In [ ]:
%%task ch07/04-imbalance/lr-bal
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Upsampling руками · база

Сделайте upsampling обучающей части:

1. Таблица `train` — `X_train` плюс столбец `churned` из `y_train`.
2. `extra` — недостающие уходящие: `sample(n=..., replace=True, random_state=0)` из уходящих гостей `train`, столько, чтобы классов стало поровну.
3. `train_up = pd.concat([train, extra])`.

Обучите на `train_up` модель `LogisticRegression(max_iter=2000)` (без весов) и сохраните `rec_up` и `prec_up` — полноту и точность на тесте (3 знака).

*Подсказки (2): `dsc.hint('upsample')`*

In [ ]:
%%task ch07/04-imbalance/upsample
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Веса у леса · база

Сравните три леса из 50 деревьев с `random_state=0`, меняя по одной настройке: обычный, с `class_weight="balanced"` и с `class_weight="balanced", min_samples_leaf=5`. Сохраните точность и полноту на тесте по ответам `predict` (3 знака): `prec_plain`, `rec_plain`, `prec_w`, `rec_w`, `prec_bal`, `rec_bal`.

*Подсказки (2): `dsc.hint('forest-bal')`*

In [ ]:
%%task ch07/04-imbalance/forest-bal
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Утечка через балансировку · посложнее

Повторите ошибку из теории и сравните с честным подходом. Лес везде — `RandomForestClassifier(n_estimators=50, random_state=0)`.

1. **Неправильно**: таблица `full` — `X` плюс столбец `churned`; размножьте уходящих во всей таблице (`sample(n=..., replace=True, random_state=0)`, классов поровну), затем разбейте `train_test_split(..., test_size=0.25, random_state=42, stratify=...)`. Сохраните `f1_leak` — F1 леса на такой «тестовой» части (3 знака) и `share_copies` — долю уходящих гостей этой тестовой части, чей индекс встречается в её обучающей части (3 знака).
2. **Правильно**: возьмите обычное разбиение (оно уже в заготовке), размножьте уходящих только в обучающей части тем же способом и сохраните `f1_honest` — F1 леса на обычном тесте (3 знака).

*Подсказки (2): `dsc.hint('ch07/04-imbalance/leak')`*

In [ ]:
%%task ch07/04-imbalance/leak
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Веса или порог · посложнее

Проверьте, что сдвиг порога у обычной модели даёт почти то же, что веса.

1. Взвешенная модель `LogisticRegression(max_iter=2000, class_weight="balanced")`: `rec_bal` и `prec_bal` — полнота и точность на тесте при пороге 0.5 (3 знака).
2. Обычная модель `LogisticRegression(max_iter=2000)`: по её PR-кривой на тесте (`precision_recall_curve`, прошлый урок) найдите порог `t_match` с наибольшей точностью среди тех, где полнота **не ниже** `rec_bal`, и `prec_match` — точность при нём (оба — 3 знака).

Для сравнения с `rec_bal` берите неокруглённую полноту.

*Подсказки (2): `dsc.hint('thr-match')`*

In [ ]:
%%task ch07/04-imbalance/thr-match
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, precision_recall_curve

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Балансировка внутри кросс-валидации · со звёздочкой

Upsampling и кросс-валидация: где утечка? Работаем только с `X_train, y_train`, лес `RandomForestClassifier(n_estimators=50, random_state=0)`, фолды `kf = KFold(n_splits=5, shuffle=True, random_state=42)` (глава 6, урок «Честное сравнение моделей»).

1. **Неправильно**: размножьте уходящих во всей обучающей части (`train_up`, как в задаче про upsampling) и оцените лес через `cross_val_score(model, X_up, y_up, cv=kf, scoring="f1")`. Параметр `scoring` говорит, какую метрику считать на каждом фолде: по умолчанию accuracy, `"f1"` — F1. Сохраните `f1_wrong` — среднее (3 знака).
2. **Правильно**: циклом `for tr, va in kf.split(X_train):` размножайте уходящих только в обучающих фолдах (`train.iloc[tr]`, тот же `sample(..., random_state=0)`) — столько, чтобы внутри обучающего фолда классов стало поровну, обучайте лес и считайте F1 на валидационном фолде **без** размножения. Сохраните `f1_right` — среднее по фолдам (3 знака).

*Подсказки (2): `dsc.hint('cv-up')` · задача по желанию*

In [ ]:
%%task ch07/04-imbalance/cv-up
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
train = X_train.copy()
train["churned"] = y_train
kf = KFold(n_splits=5, shuffle=True, random_state=42)

## Урок 6. Масштабирование и ближайшие соседи

В таблице гостей средний чек измеряется сотнями рублей, а жалобы — единицами. Для дерева это неважно, а для моделей, которые складывают и сравнивают числа, — очень. Учимся приводить признаки к одному масштабу без утечки и видим, как это меняет метод ближайших соседей и логистическую регрессию. Регуляризация — в следующем уроке.

Теория — на сайте курса: `ch07/05-scaling.html`. Задачи «собери код из строк» (1) остались на сайте.

### Обучить масштаб · разминка

Отмасштабируйте признаки `StandardScaler`, не допуская утечки. Сохраните:

- `Xs_train`, `Xs_test` — преобразованные обучающую и тестовую таблицы;
- `days_mean` — среднее `days_since_last`, которое запомнил масштабировщик (2 знака);
- `test_mean` — среднее стандартизированного `days_since_last` на тесте (3 знака). Столбец `days_since_last` — четвёртый, индекс 3; его номер можно не считать глазами: `list(X.columns).index("days_since_last")`.

*Подсказки (2): `dsc.hint('scale-fit')`*

In [ ]:
%%task ch07/05-scaling/scale-fit
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Стандартизация руками · разминка

Повторите `StandardScaler` для одного столбца без sklearn. Посчитайте по **обучающим** гостям среднее `mu` и стандартное отклонение `sd` столбца `avg_check` так, как это делает StandardScaler, и сохраните в `z_test` стандартизированный `avg_check` тестовых гостей (Series). Подсказка про отклонение — в разделе «Почему не совпадает с pandas».

*Подсказки (2): `dsc.hint('zscore')`*

In [ ]:
%%task ch07/05-scaling/zscore
import pandas as pd
from sklearn.model_selection import train_test_split

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

mu = X_train["avg_check"].mean()
sd = X_train["avg_check"].std()

### Соседи до и после · база

Обучите `KNeighborsClassifier(n_neighbors=15)` дважды: на сырых признаках и на стандартизированных (`StandardScaler`, обученный на train). Сохраните для тестовых гостей F1 при пороге по умолчанию и ROC-AUC (все — 3 знака): `f1_raw`, `auc_raw` — без масштаба, `f1_sc`, `auc_sc` — с масштабом.

*Подсказки (2): `dsc.hint('knn-scale')`*

In [ ]:
%%task ch07/05-scaling/knn-scale
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Сколько шагов до решения · база

Сравните, как обучается логистическая регрессия с масштабом и без. Сохраните:

- `it_raw` — число шагов (`n_iter_[0]`, целое) у `LogisticRegression(max_iter=2000)` на сырых признаках;
- `it_sc` — у `LogisticRegression()` на стандартизированных;
- `diff_auc` — модуль разницы их ROC-AUC на тесте (3 знака).

*Подсказки (2): `dsc.hint('iters')`*

In [ ]:
%%task ch07/05-scaling/iters
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Рейтинг признаков · база

Обучите `LogisticRegression()` на стандартизированных признаках. Сохраните:

- `imp` — Series модулей весов с названиями признаков в индексе, отсортированную по убыванию;
- `top3` — список из трёх самых сильных признаков;
- `raw_place` — какое место (1 — первое) занимает `days_since_last` по модулю веса у модели `LogisticRegression(max_iter=2000)` на **сырых** признаках.

*Подсказки (2): `dsc.hint('coef-rank')`*

In [ ]:
%%task ch07/05-scaling/coef-rank
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Кому важен масштаб · посложнее

Проверьте на деле, какие модели замечают масштабирование. Обучите каждую модель дважды — на сырых и на стандартизированных признаках — и сравните прогнозы `predict` на тесте. Сохраните долю тестовых гостей, для которых прогнозы совпали (3 знака):

- `same_tree` — для `DecisionTreeClassifier(max_depth=5, random_state=0)`;
- `same_knn` — для `KNeighborsClassifier(n_neighbors=15)`.

*Подсказки (2): `dsc.hint('who-cares')`*

In [ ]:
%%task ch07/05-scaling/who-cares
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

### Соседи руками · со звёздочкой

Повторите метод ближайших соседей без `KNeighborsClassifier`. Для **первых пяти** тестовых гостей (строки 0–4 массива `Xs_test`) найдите 15 ближайших обучающих гостей по обычному (евклидову) расстоянию в стандартизированных признаках и посчитайте долю ушедших среди них. Сохраните список из пяти долей в `hand`. Затем сравните с моделью: в `model_p` — список вероятностей ухода `KNeighborsClassifier(n_neighbors=15)` для тех же пяти гостей.

*Подсказки (2): `dsc.hint('knn-hand')` · задача по желанию*

In [ ]:
%%task ch07/05-scaling/knn-hand
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

hand = []

## Урок 7. Регуляризация: штраф за большие веса

Линейная модель может раздуть вес признака, который на обучающих данных случайно хорошо работает. Регуляризация штрафует большие веса: разбираемся, как параметр C держит веса в узде, почему он работает «наоборот» и как штраф L1 сам отбирает признаки.

Теория — на сайте курса: `ch07/05b-regularization.html`.

### Куда крутить C · разминка

Обучите на стандартизированных признаках две модели: `LogisticRegression(C=0.01)` и `LogisticRegression(C=10)`. Сохраните суммы модулей их весов (`np.abs(model.coef_).sum()`, 2 знака) в `sum_small` (для `C=0.01`) и `sum_big` (для `C=10`). В `bigger` запишите число `C` той модели, у которой сумма больше: `0.01` или `10`.

Сначала предскажите ответ, потом проверьте кодом.

*Подсказки (2): `dsc.hint('c-two')`*

In [ ]:
%%task ch07/05b-regularization/c-two
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
Xs_train = scaler.transform(X_train)

### Сила регуляризации · база

Для `C` из списка `[0.001, 0.01, 0.1, 1, 10]` обучите `LogisticRegression(C=C)` на стандартизированных признаках. Соберите таблицу `res` с индексом — значениями `C` (в том же порядке) и столбцами:

- `coef_sum` — сумма модулей весов;
- `auc` — ROC-AUC на тесте;
- `f1` — F1 на тесте при пороге по умолчанию.

Округлять не нужно. В `best_c` сохраните `C` с наибольшим F1.

*Подсказки (2): `dsc.hint('c-sweep')`*

In [ ]:
%%task ch07/05b-regularization/c-sweep
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

rows = []
for C in [0.001, 0.01, 0.1, 1, 10]:
    pass

### Три признака вместо пятнадцати · посложнее

Обучите логистическую регрессию с L1-штрафом и `C=0.01` (`solver="liblinear"`) на стандартизированных признаках. Сохраните:

- `kept` — список названий признаков с ненулевым весом (в порядке столбцов `X`);
- `auc_l1` — ROC-AUC этой модели на тесте (3 знака);
- `auc_l2` — ROC-AUC `LogisticRegression()` на тех же признаках (3 знака).

*Подсказки (2): `dsc.hint('lasso')`*

In [ ]:
%%task ch07/05b-regularization/lasso
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
scaler = StandardScaler().fit(X_train)
Xs_train, Xs_test = scaler.transform(X_train), scaler.transform(X_test)

## Урок 8. Новые признаки

Модель знает о гостях только то, что лежит в столбцах таблицы. Часто самое полезное знание аналитика — «новички уходят чаще», «жалоба на доставку — плохой знак» — модель сама не найдёт. Учимся превращать такие наблюдения в новые признаки — флаги и взаимодействия — и сравнивать варианты кросс-валидацией на одних и тех же фолдах. Как проверить, что прирост не шум, и какие признаки на самом деле важны, — в следующем уроке.

Теория — на сайте курса: `ch07/06-features.html`.

### Флаг новичка · разминка

Добавьте в таблицу `g` столбец `is_new`: 1, если гость с нами 3 месяца или меньше, иначе 0 (целые числа). Сохраните долю ушедших среди новичков `rate_new` и среди остальных `rate_old` (обе — 3 знака) и число новичков `n_new`.

*Подсказки (2): `dsc.hint('is-new')`*

In [ ]:
%%task ch07/06-features/is-new
import pandas as pd

g = pd.read_csv("guests_churn.csv")

### Жалобы на доставку · разминка

Добавьте в `g` столбец `compl_x_deliv` — произведение `complaints` на `delivery_share`. Сохраните:

- `n_hi` — сколько гостей с `compl_x_deliv` ≥ 0.5;
- `rate_hi` — долю ушедших среди них (3 знака);
- `rate_zero` — долю ушедших среди гостей с `compl_x_deliv` = 0 (3 знака).

*Подсказки (2): `dsc.hint('cxd')`*

In [ ]:
%%task ch07/06-features/cxd
import pandas as pd

g = pd.read_csv("guests_churn.csv")

### Что даёт признак · база

Функция `cv_auc` из урока уже есть в заготовке. Посчитайте средний CV-AUC (3 знака):

- `auc_base` — на исходной таблице;
- `auc_vpm` — с добавленным столбцом `visits_per_month` (визиты за 90 дней / 3);
- `auc_new` — с добавленным флагом `is_new` (без `visits_per_month`).

Каждый вариант добавляйте к своей копии таблицы (`g.copy()`), чтобы признаки не смешивались.

*Подсказки (2): `dsc.hint('ch07/06-features/gain')`*

In [ ]:
%%task ch07/06-features/gain
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(df, scoring="roc_auc"):
    X = pd.get_dummies(df.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    Xs = StandardScaler().fit_transform(X_train)
    return cross_val_score(LogisticRegression(max_iter=2000), Xs, y_train, cv=kf, scoring=scoring).mean()

### Два признака: CV, потом тест · посложнее

Соберите оба признака урока в одной копии таблицы: `f = g.copy()`, в ней `is_new` (стаж ≤ 3 месяцев, 0/1) и `compl_x_deliv` (жалобы × доля доставки). Сравните с исходной таблицей кросс-валидацией на одних фолдах (функция `cv_auc` уже в заготовке), все числа — 3 знака:

- `auc_base`, `f1_base` — CV-AUC и CV-F1 на исходной `g`;
- `auc_both`, `f1_both` — то же на `f`.

Признаки выбраны по CV — теперь один раз проверьте на тесте. Соберите из `f` таблицу признаков так же, как внутри `cv_auc` (`get_dummies`, `fillna`, то же `train_test_split`), обучите `StandardScaler` на обучающей части и `LogisticRegression(max_iter=2000)` на отмасштабированной обучающей части. Сохраните `test_auc` (по `predict_proba`) и `test_f1` (по `predict`) на тесте.

*Подсказки (2): `dsc.hint('both')`*

In [ ]:
%%task ch07/06-features/both
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, f1_score

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(df, scoring="roc_auc"):
    X = pd.get_dummies(df.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    Xs = StandardScaler().fit_transform(X_train)
    return cross_val_score(LogisticRegression(max_iter=2000), Xs, y_train, cv=kf, scoring=scoring).mean()

### Поиск взаимодействий перебором · со звёздочкой

Взаимодействие жалоб и доставки мы нашли по сводной таблице. А можно ли найти его перебором? Для всех пар столбцов из списка `cols` добавьте к копии таблицы произведение пары (один новый столбец) и посчитайте прирост среднего CV-AUC относительно исходной таблицы (функция `cv_auc` уже есть). Сохраните:

- `gains` — Series приростов; индекс — строки вида `"a*b"`, где `a` и `b` — названия столбцов в порядке, в котором их выдаёт `itertools.combinations(cols, 2)`; отсортирована по убыванию;
- `best` — индекс лучшей пары (строка `"a*b"`).

*Подсказки (2): `dsc.hint('ch07/06-features/pairs')` · задача по желанию*

In [ ]:
%%task ch07/06-features/pairs
import itertools
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(df, scoring="roc_auc"):
    X = pd.get_dummies(df.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    Xs = StandardScaler().fit_transform(X_train)
    return cross_val_score(LogisticRegression(max_iter=2000), Xs, y_train, cv=kf, scoring=scoring).mean()

cols = ["tenure_months", "app", "visits_90d", "days_since_last", "delivery_share", "promo_used", "complaints"]
print(list(itertools.combinations(cols, 2))[:3])

## Урок 9. Форма связи и важность признаков

Признак придуман — но помогает ли он на самом деле? Учимся подсказывать модели форму связи логарифмом и интервалами, отличать устойчивый прирост от шума сравнением по фолдам, честно мерить важность признаков перестановкой и ловить признаки из будущего.

Теория — на сайте курса: `ch07/06b-feature-check.html`. Задачи «собери код из строк» (1) остались на сайте.

### Интервалы давности в модели · база

Добавьте в копию таблицы столбец `recency` — давность визита по интервалам `[-1, 14, 30, 60, 1000]` (названия интервалов — любые). Посчитайте `auc_bins` — средний CV-AUC с этим столбцом (3 знака) и `n_cols` — сколько столбцов получается в таблице признаков после `get_dummies` (столько признаков видит модель).

*Подсказки (2): `dsc.hint('bins')`*

In [ ]:
%%task ch07/06b-feature-check/bins
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(df, scoring="roc_auc"):
    X = pd.get_dummies(df.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    Xs = StandardScaler().fit_transform(X_train)
    return cross_val_score(LogisticRegression(max_iter=2000), Xs, y_train, cv=kf, scoring=scoring).mean()

b = g.copy()

### Три признака вместе · база

Добавьте в копию таблицы три признака: `is_new` (стаж ≤ 3 месяцев, 0/1), `compl_x_deliv` (жалобы × доля доставки) и `log_days` (`np.log1p` от давности визита). Посчитайте средний CV-AUC `auc_all` и средний CV-F1 `f1_all` (оба — 3 знака), а также прирост F1 по сравнению с исходной таблицей `f1_gain` (3 знака, считайте по неокруглённым значениям).

*Подсказки (2): `dsc.hint('ch07/06b-feature-check/combo')`*

In [ ]:
%%task ch07/06b-feature-check/combo
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_auc(df, scoring="roc_auc"):
    X = pd.get_dummies(df.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
    Xs = StandardScaler().fit_transform(X_train)
    return cross_val_score(LogisticRegression(max_iter=2000), Xs, y_train, cv=kf, scoring=scoring).mean()

### Кому нужны новые признаки · посложнее

Сравните, сколько дают три признака из задачи «Три признака вместе» (`is_new`, `compl_x_deliv`, `log_days`) двум моделям. Посчитайте средний CV-AUC на обучающей части (одни и те же фолды `kf`) для таблиц без и с новыми признаками:

- логистическая регрессия — как в `cv_auc` (масштаб + `LogisticRegression(max_iter=2000)`);
- лес — `RandomForestClassifier(n_estimators=50, random_state=0)` без масштабирования.

Сохраните приросты `lr_gain` и `rf_gain` (новые минус исходные, 3 знака, по неокруглённым средним) и `rf_std` — разброс (`std`) AUC леса по фолдам на исходной таблице (3 знака).

*Подсказки (2): `dsc.hint('rf-gain')`*

In [ ]:
%%task ch07/06b-feature-check/rf-gain
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
kf = KFold(n_splits=5, shuffle=True, random_state=42)

new = g.copy()
new["is_new"] = (new["tenure_months"] <= 3).astype(int)
new["compl_x_deliv"] = new["complaints"] * new["delivery_share"]
new["log_days"] = np.log1p(new["days_since_last"])

### Важность перестановкой · база

Обучите `RandomForestClassifier(n_estimators=50, random_state=0)` на обучающей части и посчитайте важность перестановкой на **тесте**: `scoring="roc_auc"`, `n_repeats=5`, `random_state=0`. Сохраните:

- `pi_s` — Series средних падений AUC с названиями признаков в индексе, отсортированную по убыванию;
- `top3` — список трёх самых важных признаков;
- `check_pi` — важность `avg_check` по перестановке (3 знака);
- `check_place` — место `avg_check` (1 — первое) по встроенной `feature_importances_` того же леса.

*Подсказки (2): `dsc.hint('perm')`*

In [ ]:
%%task ch07/06b-feature-check/perm
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

### Шум в рейтинге · посложнее

В заготовке к признакам добавлен столбец случайного шума `noise`. Обучите `RandomForestClassifier(n_estimators=100, random_state=0)` и сравните два рейтинга важности. Сохраните место столбца `noise` (1 — первое): `noise_fi` — по `feature_importances_`, `noise_pi` — по важности перестановкой на тесте (`scoring="roc_auc"`, `n_repeats=5`, `random_state=0`). В `below_noise` — список признаков, которые по `feature_importances_` стоят **ниже** шума (в порядке рейтинга).

*Подсказки (2): `dsc.hint('noise-rank')`*

In [ ]:
%%task ch07/06b-feature-check/noise-rank
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

g = pd.read_csv("guests_churn.csv")
y = g["churned"]
X = pd.get_dummies(g.drop(columns=["guest_id", "churned", "card_blocked"]), dtype=int).fillna({"avg_check": 0})
X["noise"] = np.random.RandomState(0).rand(len(X))
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

## Урок 10. Pipeline: предобработка и модель вместе

До сих пор мы готовили данные руками: `get_dummies`, `fillna`, масштабирование — и повторяли то же самое для теста и новых гостей. Соберём всю подготовку и модель в один объект-конвейер: он сам учится на обучающей части, одинаково обрабатывает любые новые строки и честно проходит кросс-валидацию.

Теория — на сайте курса: `ch07/07-pipeline.html`. Задачи «собери код из строк» (1) остались на сайте.

### Медиана из обучения · разминка

`X_train` и `X_test` (сырые, с пропусками в `avg_check`) уже готовы. Создайте `imp = SimpleImputer(strategy="median")`, обучите его на `X_train[["avg_check"]]` и заполните пропуски в `X_test[["avg_check"]]`. Сохраните:

- `med` — медиану, которую запомнил заполнитель (число из `imp.statistics_`);
- `test_filled` — результат `transform` для теста.

*Подсказки (2): `dsc.hint('median-imp')`*

In [ ]:
%%task ch07/07-pipeline/median-imp
from sklearn.impute import SimpleImputer

### Флаги для новых городов · разминка

`X_train` уже готов. Обучите `OneHotEncoder(handle_unknown="ignore", sparse_output=False)` на столбце `X_train[["city"]]`. Закодируйте таблицу `new_cities` (готова, в ней Уфа, Екатеринбург и Пермь) и сохраните результат в `codes`, а имена столбцов (`get_feature_names_out()`) списком — в `names`. В `unknown_rows` — сколько строк `codes` целиком из нулей.

*Подсказки (2): `dsc.hint('ohe-new')`*

In [ ]:
%%task ch07/07-pipeline/ohe-new
from sklearn.preprocessing import OneHotEncoder

### Что получается после подготовки · база

Сырые `X_train`, `X_test` и списки `num`, `cat` уже готовы. Соберите `pre` — ColumnTransformer как в теории (числовые: заполнить нулём и масштабировать; категориальные: `OneHotEncoder(handle_unknown="ignore")`). Обучите его на `X_train` и преобразуйте обе части: `Xt_train`, `Xt_test`. Сохраните:

- `n_cols` — сколько столбцов получилось;
- `names` — список имён столбцов (`get_feature_names_out()`);
- `test_mean` — среднее по тесту в столбце `num__days_since_last` (2 знака). У обучающей части среднее этого столбца ровно 0 — а у теста?

*Подсказки (2): `dsc.hint('prep-shape')`*

In [ ]:
%%task ch07/07-pipeline/prep-shape
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

### Конвейер на тесте · база

Сырые `X_train`, `X_test`, `y_train`, `y_test` и готовый (не обученный) ColumnTransformer `pre` уже есть. Соберите `pipe = Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=2000))])`, обучите на обучающей части и посчитайте на тесте `test_f1` и `test_auc` (3 знака). В `top_feature` сохраните имя признака с наибольшим по модулю коэффициентом модели (имена — из шага `prep`).

*Подсказки (2): `dsc.hint('pipe-metrics')`*

In [ ]:
%%task ch07/07-pipeline/pipe-metrics
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score

### Риск для новых гостей · база

Конвейер `pipe` (подготовка + логистическая регрессия) уже обучен на всех гостях. Пришла выгрузка `new` — 5 новых гостей в «сыром» виде: среди них гость из нового города Екатеринбург и гости с пустым средним чеком.

Сохраните в `risk` вероятности ухода для этих гостей (список, 3 знака), в `flag` — предсказания 0/1 (список), а в `riskiest` — `guest_id` гостя с наибольшим риском. Столбец `guest_id` в конвейер подавать не нужно, но и удалять его не обязательно — проверьте, что получится.

*Подсказки (2): `dsc.hint('new-guests')`*

In [ ]:
%%task ch07/07-pipeline/new-guests
# pipe уже обучен, new — сырая таблица новых гостей

### Одна подготовка — три модели · посложнее

Сырые `X_train`, `y_train`, ColumnTransformer `pre` и фолды `kf = KFold(n_splits=5, shuffle=True, random_state=42)` уже готовы. Сравните кросс-валидацией по `scoring="roc_auc"` три конвейера с одной и той же подготовкой `pre`:

- `"логрегрессия"` — `LogisticRegression(max_iter=2000)`;
- `"соседи"` — `KNeighborsClassifier(n_neighbors=25)`;
- `"лес"` — `RandomForestClassifier(n_estimators=50, random_state=0)`.

Сохраните словарь `cv_auc`: название → средний AUC по фолдам (3 знака) и `best_model` — название лучшего.

*Подсказки (2): `dsc.hint('cv-models')`*

In [ ]:
%%task ch07/07-pipeline/cv-models
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
cv_auc = {}

## Урок 11. Свои шаги конвейера: порядок категорий, функции и свой класс

Готовых кодировщиков и заполнителей хватает не всегда. Учимся передавать конвейеру порядок категорий, встраивать в него свои признаки функцией и писать собственный шаг-трансформер — класс с fit и transform, который ведёт себя как родной для sklearn.

Теория — на сайте курса: `ch07/07b-own-steps.html`. Задачи «собери код из строк» (1) остались на сайте.

### Кофемашина · разминка

Допишите класс `CoffeeMachine` — кофемашину, которая помнит запас зёрен и число сваренных чашек:

- `__init__(self, beans)` запоминает запас зёрен в граммах в атрибуте `beans` и заводит счётчик `cups = 0`;
- `brew(self, grams=18)` уменьшает запас на `grams`, увеличивает счётчик на 1 и возвращает сам объект (`self`).

Затем создайте `m = CoffeeMachine(1000)`, сварите на ней три чашки по 18 г и сохраните в `left` остаток зёрен, а в `cups` — число чашек (из атрибутов `m`).

*Подсказки (2): `dsc.hint('class-warm')`*

In [ ]:
%%task ch07/07b-own-steps/class-warm
class CoffeeMachine:
    def __init__(self, beans):
        # запомните запас в self.beans и заведите счётчик self.cups
        pass

    def brew(self, grams=18):
        # уменьшите запас, увеличьте счётчик, верните self
        pass

m = CoffeeMachine(1000)

### Уровень карты по порядку · база

Сырые `X_train`, `y_train` (уже с уровнем карты `card_tier`) и список `TIERS = ["базовая", "серебряная", "золотая"]` готовы. Обучите на `X_train[["card_tier"]]` порядковый кодировщик `enc` с этим порядком, который ставит −1 вместо неизвестного уровня. Сохраните:

- `codes` — коды уровня для `X_train` плоским массивом;
- `rate_by_code` — список долей ушедших (`y_train`) для кодов 0, 1, 2 по порядку (3 знака);
- `new_code` — код, который `enc` даёт уровню «платиновая» (число).

*Подсказки (3): `dsc.hint('ord-tier')`*

In [ ]:
%%task ch07/07b-own-steps/ord-tier
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

### Признаки в конвейере · посложнее

Сырые `X_train`, `y_train` и `kf` уже готовы. Напишите функцию `add_features(df)`, которая возвращает **копию** таблицы с двумя новыми столбцами:

- `is_new` — 1, если `tenure_months <= 3`, иначе 0;
- `compl_x_deliv` — `complaints * delivery_share`.

Соберите конвейер `pipe`: шаг `"feat"` — `FunctionTransformer(add_features)`, шаг `"prep"` — ColumnTransformer как в теории, но с новыми столбцами в числовом списке, шаг `"model"` — `LogisticRegression(max_iter=2000)`. Посчитайте кросс-валидацией на `X_train` средний F1 (`scoring="f1"`, `cv=kf`): `f1_feat` — для нового конвейера и `f1_base` — для конвейера без шага `feat` и без новых столбцов. Оба — 3 знака.

*Подсказки (2): `dsc.hint('feat-step')`*

In [ ]:
%%task ch07/07b-own-steps/feat-step
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression

num = ["tenure_months", "app", "visits_90d", "days_since_last", "avg_check",
       "delivery_share", "promo_used", "complaints"]
cat = ["city", "fav_category"]

def add_features(df):
    return df

### Визиты относительно своей категории · посложнее

Сырые `X_train` и `X_test` уже готовы. Напишите трансформер `DiffFromGroup(col, by)` с родителями `TransformerMixin` и `BaseEstimator` (в этом порядке):

- `fit` запоминает средние `col` по значениям `by` (`means_`) и общее среднее `col` (`overall_`) — только по поданной таблице;
- `transform` возвращает **копию** таблицы с новым столбцом `f"{col}_minus_{by}"` = значение `col` минус выученное среднее его группы; для группы, которой не было при `fit`, — минус общее среднее.

Обучите `t = DiffFromGroup(col="visits_90d", by="fav_category")` на `X_train` и сохраните `out = t.transform(X_test)`.

*Подсказки (3): `dsc.hint('group-ratio')`*

In [ ]:
%%task ch07/07b-own-steps/group-ratio
import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin

class DiffFromGroup(TransformerMixin, BaseEstimator):
    def __init__(self, col="visits_90d", by="fav_category"):
        ...

    def fit(self, X, y=None):
        ...

    def transform(self, X):
        ...

### Свой признак · со звёздочкой

Сырые `X_train`, `y_train`, `kf` и функция `add_features` из прошлой задачи (флаг новичка и жалобы × доставка) уже готовы. Конвейер с ней даёт средний CV-AUC около 0.905.

Напишите функцию `add_more(df)`: она вызывает `add_features(df)` и добавляет **один** новый числовой столбец с именем `extra`. Добейтесь, чтобы CV-AUC конвейера (`scoring="roc_auc"`, `cv=kf`, числовые столбцы — 8 исходных + `is_new`, `compl_x_deliv`, `extra`) вырос хотя бы на 0.004 по сравнению с конвейером без `extra` (это примерно столько, сколько дал лучший признак в уроке «Форма связи и важность признаков», где он выигрывал на всех фолдах). В `auc_more` сохраните получившийся средний AUC (3 знака).

Признак должен считаться по строке (никаких `.mean()` по столбцу) и не использовать `churned`.

*Подсказки (2): `dsc.hint('own-feature')` · задача по желанию*

In [ ]:
%%task ch07/07b-own-steps/own-feature
from sklearn.model_selection import cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression

def add_more(df):
    df = add_features(df)
    df["extra"] = 0
    return df

## Урок 12. Подбор гиперпараметров

У модели и подготовки есть настройки, которые не выучиваются из данных: сила регуляризации, веса классов, глубина деревьев. Учимся перебирать их честно и автоматически — `GridSearchCV` по всему конвейеру, — читать таблицу результатов, один раз проверять итог на тесте и сохранять готовую модель в файл.

Теория — на сайте курса: `ch07/08-tuning.html`. Задачи «собери код из строк» (1) остались на сайте.

### Сетка для конвейера · разминка

Конвейер `pipe` (шаги `"prep"` и `"model"` — логистическая регрессия) уже готов. Составьте сетку `param_grid`: параметр `C` модели — значения `0.1, 1, 10`, веса классов `class_weight` — `None` и `"balanced"`.

В `n_fits` сохраните, сколько обучений сделает `GridSearchCV` с этой сеткой и `cv=kf` (5 фолдов) — без итогового обучения на всей обучающей части. Посчитайте через длины списков в сетке, а не вписывайте число руками.

*Подсказки (2): `dsc.hint('grid-keys')`*

In [ ]:
%%task ch07/08-tuning/grid-keys
param_grid = {
    "C": [0.1, 1, 10],
}
n_fits = 0

### Сохранить и загрузить · разминка

Конвейер `pipe` уже обучен на `X_train`. Сохраните его в файл `churn_model.joblib`, загрузите обратно в переменную `model2` и проверьте, что ответы не изменились:

- `same` — `True`, если `model2.predict(X_test)` совпадает с `pipe.predict(X_test)` во всех строках;
- `n_flag` — сколько гостей теста загруженная модель относит к уходящим (ответ 1).

*Подсказки (2): `dsc.hint('save-load')`*

In [ ]:
%%task ch07/08-tuning/save-load
import joblib
model2 = pipe

### Подбор для логистической регрессии · база

`pipe`, `kf`, `X_train`, `y_train` уже готовы. Переберите `GridSearchCV` по сетке: `C` — `0.01, 0.1, 1, 10`, `class_weight` — `None` и `"balanced"`; метрика — F1, фолды — `kf`. Сохраните:

- `search` — обученный поиск;
- `best_params` — лучшие настройки (`best_params_`);
- `best_f1` — их средний F1 (3 знака);
- `gain` — насколько лучший средний F1 выше, чем у настроек по умолчанию (`C=1`, без весов), 3 знака. Средний F1 настроек по умолчанию возьмите из `cv_results_`.

*Подсказки (3): `dsc.hint('grid-lr')`*

In [ ]:
%%task ch07/08-tuning/grid-lr
from sklearn.model_selection import GridSearchCV

### Читаем таблицу перебора · база

Поиск `search` (сетка `C` × `class_weight`, метрика F1) уже обучен. Соберите из `search.cv_results_` таблицу `table` со столбцами `param_model__C`, `param_model__class_weight`, `mean_test_score`, `std_test_score`, отсортированную по `rank_test_score` (лучшее сочетание — первой строкой).

В `n_close` сохраните, у скольких сочетаний (включая лучшее) средний F1 не ниже, чем «F1 лучшего минус его `std_test_score`», — то есть сколько сочетаний неотличимы от лучшего в пределах разброса. В `cw_close` — множество (`set`) значений `class_weight` у этих сочетаний.

*Подсказки (3): `dsc.hint('cv-table')`*

In [ ]:
%%task ch07/08-tuning/cv-table
import pandas as pd
res = pd.DataFrame(search.cv_results_)

### Один раз на тесте · база

Поиск `search` (сетка `C` × `class_weight`, метрика F1) уже обучен на `X_train`. Пора проверить итог. Сохраните:

- `final` — конвейер с лучшими настройками, обученный на всей обучающей части (его готовит сам поиск);
- `test_f1` и `test_auc` — F1 и ROC-AUC `final` на тесте (3 знака);
- `cv_gap` — `test_f1` минус средний F1 лучшего сочетания на кросс-валидации, 3 знака.

*Подсказки (2): `dsc.hint('final-test')`*

In [ ]:
%%task ch07/08-tuning/final-test
from sklearn.metrics import f1_score, roc_auc_score
final = pipe

### Настройка подготовки · посложнее

`pipe`, `kf`, `X_train`, `y_train` уже готовы. Перебирать можно не только параметры модели, но и параметры подготовки. Проверим гипотезу: может, пропуски в числовых столбцах лучше заполнять медианой, а не нулём?

Соберите сетку: стратегия заполнителя в числовой части подготовки — `"constant"` и `"median"`; `C` модели — `0.1` и `1`; `class_weight` — только `"balanced"`. Обучите `search` по F1 с `cv=kf` и сохраните:

- `imp_key` — имя параметра стратегии заполнителя (строка, как в сетке);
- `best_params` — лучшие настройки;
- `delta` — насколько лучший средний F1 со стратегией `"median"` выше лучшего среднего F1 со стратегией `"constant"` (может быть и отрицательным), 3 знака.

*Подсказки (3): `dsc.hint('prep-params')`*

In [ ]:
%%task ch07/08-tuning/prep-params
from sklearn.model_selection import GridSearchCV
imp_key = "strategy"

### Случайный поиск для леса · посложнее

`pre`, `kf`, `X_train`, `y_train` уже готовы. Соберите конвейер `forest`: шаг `"prep"` — `pre`, шаг `"model"` — `RandomForestClassifier(n_estimators=50, random_state=0)`. Запустите `RandomizedSearchCV` по F1 с `cv=kf`, `n_iter=5`, `random_state=0` и распределением:

- `max_depth` — `3, 5, 8, None`;
- `min_samples_leaf` — `1, 3, 5, 10`;
- `class_weight` — `None`, `"balanced"`.

Сохраните `rs` (обученный поиск), `best_params` и `best_f1` (3 знака). В `beats_lr` — `True`, если лес лучше логистической регрессии с весами классов: её средний CV-F1 уже лежит в `lr_f1`.

*Подсказки (2): `dsc.hint('random-forest')`*

In [ ]:
%%task ch07/08-tuning/random-forest
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

### Своя метрика: топ-30 % · со звёздочкой

Отдел лояльности может дать акцию удержания только 30 % гостей — самым рискованным по модели. Важно одно: какую долю всех уходящих гостей поймают эти 30 %. Сделайте эту долю метрикой подбора.

Напишите функцию `top30_recall(model, X, y)`:

1. вероятности ухода `p = model.predict_proba(X)[:, 1]`;
2. `k = int(len(p) * 0.3)` гостей с наибольшими `p`;
3. вернуть, какая доля всех ушедших (`y == 1`) попала в эти `k`.

Передайте функцию в `GridSearchCV` как `scoring=top30_recall` (без кавычек и скобок) и переберите `C` — `0.01, 0.1, 1` и `class_weight` — `None`, `"balanced"` с `cv=kf`. Сохраните `search`, `best_params` и `best_top30` (3 знака).

*Подсказки (3): `dsc.hint('top30')` · задача по желанию*

In [ ]:
%%task ch07/08-tuning/top30
import numpy as np
from sklearn.model_selection import GridSearchCV

def top30_recall(model, X, y):
    return 0

## Урок 13. Проект: какие заявки довести до договора

Финал главы. У «Зерна» есть кофе-кейтеринг для компаний, и заявок больше, чем менеджеры успевают вести. Строим модель-приоритизатор: от осмотра данных и утечки до конвейера, подбора настроек и ответа на вопрос коммерческого директора — сколько договоров окажется в верхних 30 % списка.

Теория — на сайте курса: `ch07/09-project.html`.

### Шаг 1. Осмотр · база

Загрузите `catering_leads.csv` в `d`. Сохраните:

- `conv_rate` — долю заявок с договором (3 знака);
- `n_contracts` — сколько всего договоров;
- `budget_na` — долю заявок с пустым `budget_pp` (3 знака);
- `size_na` — сколько заявок с пустым `company_size`;
- `contract_match` — `True`, если `contract_rub` заполнен **ровно** у тех заявок, где `converted == 1` (и пуст у всех остальных).

*Подсказки (2): `dsc.hint('ch07/09-project/look')`*

In [ ]:
%%task ch07/09-project/look
import pandas as pd

### Шаг 2. Кто покупает · база

`d` уже загружена. Посчитайте конверсию (долю `converted == 1`):

- `by_channel` — Series: конверсия по каналам `channel`, 3 знака, по убыванию;
- `repeat_rate` и `new_rate` — у повторных (`repeat_client == 1`) и новых клиентов (3 знака);
- `fast_rate` — у заявок, где первый ответ пришёл за час или быстрее (`response_hours <= 1`), и `slow_rate` — где ответа ждали больше 8 часов (3 знака).

*Подсказки (2): `dsc.hint('groups')`*

In [ ]:
%%task ch07/09-project/groups
# d — таблица заявок

### Шаг 3. Что значат пропуски · база

`d` уже загружена. Прежде чем заполнять пропуски, посмотрим, случайны ли они. Сохраните конверсию (3 знака):

- `size_na_rate` — у заявок с пустым `company_size`, `size_ok_rate` — с заполненным;
- `budget_na_rate` — с пустым `budget_pp`, `budget_ok_rate` — с заполненным;
- `budget_high` — среди заявок с бюджетом **от 450 ₽** на гостя, `budget_low` — с бюджетом меньше 450 ₽ (пустой бюджет не входит ни туда, ни туда).

*Подсказки (2): `dsc.hint('ch07/09-project/missing')`*

In [ ]:
%%task ch07/09-project/missing
# d — таблица заявок

### Шаг 4. Признаки из времени заявки · база

`d` уже загружена. В `created` — дата и время заявки строкой. Сделайте из неё три признака и **добавьте их в `d`**:

- `hour` — час заявки (`created.dt.hour`);
- `weekday` — день недели (`created.dt.dayofweek`, 0 — понедельник);
- `evening` — 1, если заявка пришла в 19:00 или позже, иначе 0 (целые числа).

Затем сохраните:

- `evening_share` — долю вечерних заявок (3 знака);
- `evening_rate` и `day_rate` — конверсию вечерних и остальных заявок (3 знака);
- `evening_wait` и `day_wait` — медиану `response_hours` у вечерних и остальных заявок (1 знак).

*Подсказки (2): `dsc.hint('time')`*

In [ ]:
%%task ch07/09-project/time
import pandas as pd
created = pd.to_datetime(d["created"])

### Шаг 5. Конвейер · база

`d` с признаками из шага 4 уже готова. Соберите:

- `y` — `converted`; `X` — все столбцы `d`, кроме `lead_id`, `created`, `contract_rub` и `converted` (без `get_dummies` — подготовку сделает конвейер);
- `X_train`, `X_test`, `y_train`, `y_test` — `train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)`;
- `pre` — `ColumnTransformer` (урок «Pipeline: предобработка и модель вместе»): числовые столбцы `num` — `SimpleImputer(strategy="median")`, затем `StandardScaler()`; категориальные `cat` — `SimpleImputer(strategy="constant", fill_value="не указан")` (`SimpleImputer` умеет заполнять и текст — константой: пропуск станет отдельной категорией «не указан»), затем `OneHotEncoder(handle_unknown="ignore")`;
- `pipe` — `Pipeline([("prep", pre), ("model", LogisticRegression(max_iter=2000))])`.

Сохраните `n_features` — сколько столбцов получается после `pre` на обучающей части, и `cv_auc` — средний ROC-AUC конвейера на фолдах `kf` по обучающей части (3 знака).

*Подсказки (2): `dsc.hint('ch07/09-project/pipe')`*

In [ ]:
%%task ch07/09-project/pipe
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
kf = KFold(n_splits=5, shuffle=True, random_state=42)
num = ["guests", "budget_pp", "days_to_event", "response_hours", "repeat_client", "tasting",
       "hour", "weekday", "evening"]
cat = ["channel", "event_type", "city", "company_size", "manager"]

### Шаг 6. Ловушка дисбаланса · посложнее

Курс подготовил `X_train`, `y_train`, фолды `kf` и необученный `pre` из шага 5. Посчитайте на фолдах `kf` по обучающей части (все — среднее, 3 знака):

- `dummy_acc` — долю правильных `DummyClassifier(strategy="most_frequent")`;
- `forest_acc` и `forest_f1` — долю правильных и F1 конвейера `pre` + `RandomForestClassifier(n_estimators=50, min_samples_leaf=5, random_state=0)`;
- `balanced_f1` — F1 того же конвейера, но с лесом `class_weight="balanced"`.

*Подсказки (2): `dsc.hint('trap')`*

In [ ]:
%%task ch07/09-project/trap
from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier

### Шаг 7. Метрика директора · база

Напишите функцию `catch_rate(y_true, scores, share=0.3)` — долю всех договоров, которые попадают в верхние `share` заявок по оценке:

1. сделайте `pd.Series(scores, index=y_true.index)`;
2. возьмите `int(len(...) * share)` заявок с наибольшей оценкой (`nlargest`);
3. верните сумму `y_true` по ним, делённую на сумму `y_true` по всем.

Затем посчитайте точку сравнения — правило директора на **обучающей** части: правило отмечает заявку, если клиент повторный **или** пришёл по рекомендации. Правило — не оценка, а отметка 0/1, и отмечает оно не 30 % заявок, поэтому `catch_rate` к нему не применяйте. Сохраните:

- `rule_share` — какую долю заявок `X_train` отмечает правило (3 знака);
- `rule_catch` — какую долю договоров `y_train` оно ловит: договоры среди отмеченных правилом заявок, делённые на все договоры `y_train` (3 знака);
- `random_catch` — `catch_rate` для случайных оценок `np.random.default_rng(0).random(len(y_train))` (3 знака).

*Подсказки (2): `dsc.hint('metric')`*

In [ ]:
%%task ch07/09-project/metric
import numpy as np
import pandas as pd

def catch_rate(y_true, scores, share=0.3):
    ...

### Шаг 8. Подбор настроек · посложнее

Курс подготовил `pipe` из шага 5 (подготовка + `LogisticRegression(max_iter=2000)`, не обучен). Подберите настройки модели `GridSearchCV` (урок «Подбор гиперпараметров») по сетке

- `C`: 0.01, 0.1, 1, 10;
- `class_weight`: `None`, `"balanced"`

с `cv=kf` и `scoring="roc_auc"` на обучающей части. Почему AUC, а не `catch_rate` из шага 7: AUC смотрит на весь порядок заявок и устойчивее на фолдах, а в топ-30 % одного фолда всего около 70 договоров — доля по ним сильно шумит. Сохраните обученный поиск в `grid`, а также:

- `best_params` — лучшие настройки (`best_params_`);
- `best_auc` — лучший средний AUC (3 знака);
- `res` — `pd.DataFrame(grid.cv_results_)`;
- `spread` — разницу между лучшим и худшим `mean_test_score` из `res` (3 знака).

*Подсказки (2): `dsc.hint('ch07/09-project/grid')`*

In [ ]:
%%task ch07/09-project/grid
import pandas as pd
from sklearn.model_selection import GridSearchCV

### Шаг 9. Сколько договоров в топ-30 % · база

Курс обучил лучший конвейер из шага 8 на всей обучающей части — он в `best`, — и определил функцию `catch_rate` из шага 7. Теперь — один раз — проверьте на тесте (3 знака):

- `test_auc` — ROC-AUC `best` на `X_test`;
- `model_catch` — `catch_rate` вероятностей `best` на тесте (топ-30 %);
- `rule_share_test` и `rule_catch_test` — какую долю тестовых заявок отмечает правило директора и какую долю договоров оно ловит;
- `model_same` — `catch_rate` модели при `share=rule_share_test`: сколько ловит модель, если вести столько же заявок, сколько отмечает правило.

*Подсказки (2): `dsc.hint('ch07/09-project/final')`*

In [ ]:
%%task ch07/09-project/final
from sklearn.metrics import roc_auc_score

### Шаг 10. Скорость ответа: причина или следствие? · со звёздочкой

Один из самых сильных сигналов модели — `response_hours`, и вывод «отвечайте быстрее — будет больше договоров» напрашивается. Проверим две вещи.

**Модель без скорости.** Если ставить приоритет **в момент прихода** заявки, скорости ответа ещё нет. Соберите такую же подготовку без `response_hours` (список `num_now`), конвейер с `LogisticRegression(max_iter=2000)`, обучите на `X_train` и сохраните `catch_now` — `catch_rate` на тесте (3 знака).

**Менеджер Олег.** У Олега конверсия ниже, чем у коллег. Сохраните по всей таблице `d` (3 знака):

- `oleg_wait` и `others_wait` — медиану `response_hours` у Олега и у остальных менеджеров (1 знак);
- `oleg_fast` и `others_fast` — конверсию заявок с ответом за 2 часа или быстрее (`response_hours <= 2`) у Олега и у остальных.

Подготовка `pre`, списки `num` и `cat` и функция `catch_rate` — уже есть.

*Подсказки (2): `dsc.hint('speed')` · задача по желанию*

In [ ]:
%%task ch07/09-project/speed
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
num_now = [c for c in num if c != "response_hours"]

## Готово

Запустите `dsc.progress()` — увидите, какие задачи этого сеанса ещё не решены. Перед тем как закрыть блокнот, полезно сделать **Kernel → Restart Kernel and Run All Cells**: если всё проходит сверху вниз, решения не зависят от случайного порядка запуска ячеек.

In [ ]:
dsc.progress()